# OAI OARSI Sub-Grades — Extraction (auto-detecting column names)

Extracts the OARSI feature gradings (osteophytes, joint-space narrowing, sclerosis)
from `KXR_SQ_BU00.txt`. This version auto-detects the osteophyte / JSN / sclerosis
columns by pattern-matching the variable names, so it works regardless of the exact
suffix convention in your particular OAI release (a previous run missed osteophytes
because their column names did not match a hard-coded guess). Output:
`oai_oarsi_labels.csv` with KL plus all available sub-features per knee.

## Setup

In [1]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import pandas as pd, numpy as np, re
DRIVE_ROOT=Path('/content/drive/MyDrive'); PROJECT=DRIVE_ROOT/'Master Thesis'
SRC_FILE=PROJECT/'oai'/'assessments'/'XR Knee Semi-Quant'/'KXR_SQ_BU00.txt'
OUT_DIR=PROJECT/'oai'; OUT_DIR.mkdir(parents=True, exist_ok=True)
assert SRC_FILE.exists(), f'File not found: {SRC_FILE}'
df=pd.read_csv(SRC_FILE, sep='|', dtype=str, low_memory=False)
print(f'Raw rows: {len(df):,} | Columns: {df.shape[1]}')

Mounted at /content/drive
Raw rows: 16,666 | Columns: 24


## Auto-detect feature columns

OAI X-ray semi-quant variables share a `V00XR` prefix. The feature family is encoded
in the middle of the name: `OST` = osteophytes, `JS` = joint-space narrowing, `SC` =
sclerosis, `ATT` = attrition, `CY` = cysts. This cell groups every `V00XR*` column by
family using substring matching, so whatever suffixes your release uses are captured
automatically. The full list is printed for transparency.

In [2]:
xr_cols=[c for c in df.columns if c.upper().startswith('V00XR')]
print(f'{len(xr_cols)} V00XR* columns total. Grouping by feature family:\n')

def family(col):
    u=col.upper().replace('V00XR','')
    if u.startswith('KL'): return 'kl'
    if 'OST' in u: return 'osteophyte'
    if u.startswith('JS') or 'JSN' in u: return 'jsn'
    if u.startswith('SC') or 'SCL' in u: return 'sclerosis'
    if 'ATT' in u: return 'attrition'
    if u.startswith('CY') or 'CYST' in u: return 'cyst'
    return 'other'

groups={}
for c in xr_cols: groups.setdefault(family(c),[]).append(c)
for fam in ['kl','osteophyte','jsn','sclerosis','attrition','cyst','other']:
    if fam in groups:
        print(f'  {fam:11s}: {groups[fam]}')

19 V00XR* columns total. Grouping by feature family:

  kl         : ['V00XRKL']
  osteophyte : ['V00XROSTM', 'V00XROSTL']
  jsn        : ['V00XRJSM', 'V00XRJSL']
  sclerosis  : ['V00XRSCFM', 'V00XRSCTM', 'V00XRSCFL', 'V00XRSCTL']
  attrition  : ['V00XRATTM', 'V00XRATTL']
  cyst       : ['V00XRCYFM', 'V00XRCYTM', 'V00XRCYFL', 'V00XRCYTL']
  other      : ['V00XROSFM', 'V00XRCHM', 'V00XROSFL', 'V00XRCHL']


## Parse all detected feature columns

Each value is encoded like ``"2: 2"`` (code before the colon). Every detected feature
column is parsed; per-family composite maxima are then computed (the maximum grade
across compartments), which are the cleaner objective targets for multi-task
training.

In [3]:
def parse_code(val):
    if pd.isna(val): return np.nan
    tok=val.split(':')[0].strip()
    try: return int(tok)
    except ValueError: return np.nan

kl_col = groups.get('kl',['V00XRKL'])[0]
df['kl']=df[kl_col].apply(parse_code)

parsed_families={}
for fam in ['osteophyte','jsn','sclerosis','attrition','cyst']:
    cols=groups.get(fam,[])
    pcols=[]
    for c in cols:
        name=f'{fam}__'+c.upper().replace('V00XR','')
        df[name]=df[c].apply(parse_code); pcols.append(name)
    if pcols: parsed_families[fam]=pcols

df['side']=df['SIDE'].str.extract(r'^(\d+):')[0].map({'1':'R','2':'L'})
df['READPRJ']=pd.to_numeric(df['READPRJ'], errors='coerce')
print('Parsed families:', {k:len(v) for k,v in parsed_families.items()})

Parsed families: {'osteophyte': 2, 'jsn': 2, 'sclerosis': 4, 'attrition': 2, 'cyst': 4}


## Build per-knee table with composite maxima

In [4]:
d15=df[df['READPRJ']==15].copy()
d15=d15.dropna(subset=['kl','side']); d15['kl']=d15['kl'].astype(int)
d15=d15[d15['kl'].between(0,4)].drop_duplicates(subset=['ID','side'])
d15['ID']=d15['ID'].str.strip()

max_cols=[]
for fam,pcols in parsed_families.items():
    mx=f'{fam}_max'; d15[mx]=d15[pcols].max(axis=1); max_cols.append(mx)

keep=['ID','side','kl']+max_cols
out=d15[keep].rename(columns={'ID':'subject_id','kl':'kl_grade'}).sort_values(['subject_id','side']).reset_index(drop=True)
print(f'Knees: {len(out):,} | columns: {list(out.columns)}')
print('\nComposite max distributions:')
for c in max_cols:
    print(f'  {c}: {out[c].value_counts(dropna=False).sort_index().to_dict()}')

Knees: 8,921 | columns: ['subject_id', 'side', 'kl_grade', 'osteophyte_max', 'jsn_max', 'sclerosis_max', 'attrition_max', 'cyst_max']

Composite max distributions:
  osteophyte_max: {0.0: 1527, 1.0: 2765, 2.0: 682, 3.0: 533, nan: 3414}
  jsn_max: {0.0: 5144, 1.0: 2247, 2.0: 1236, 3.0: 294}
  sclerosis_max: {0.0: 3102, 1.0: 1257, 2.0: 980, 3.0: 167, nan: 3415}
  attrition_max: {0.0: 5436, 1.0: 57, 2.0: 12, 3.0: 1, nan: 3415}
  cyst_max: {0.0: 5188, 1.0: 318, nan: 3415}


## Save

In [5]:
out_path=OUT_DIR/'oai_oarsi_labels.csv'
out.to_csv(out_path, index=False)
print(f'Saved -> {out_path}')
print(f'{len(out):,} knees x {out.shape[1]} columns')
got=[c.replace('_max','') for c in out.columns if c.endswith('_max')]
print('Sub-features captured:', got)
if 'osteophyte' in got: print('Osteophytes RECOVERED — the key KL feature is now present.')
else: print('NOTE: no osteophyte columns detected — check the family grouping output above.')

Saved -> /content/drive/MyDrive/Master Thesis/oai/oai_oarsi_labels.csv
8,921 knees x 8 columns
Sub-features captured: ['osteophyte', 'jsn', 'sclerosis', 'attrition', 'cyst']
Osteophytes RECOVERED — the key KL feature is now present.
